# HVAC Complaint Intelligence — Sprint 1 End-to-End Validation

Runs the full pipeline on 500 synthetic complaints:
**generate → embed → cluster → visualise → sentiment → label → trends → summary**.

Cell 9 asserts the Sprint 1 exit criteria (≥5 clusters, silhouette > 0.2, Delhi compressor spike emerging, ≥2 CRITICAL).


## Cell 1 — Setup


In [1]:
import os, sys, random
from pathlib import Path

import numpy as np
import pandas as pd

# Make the hvac-ml package importable when running from notebooks/
REPO_ROOT = Path.cwd().resolve().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from data.generators.synthetic_complaints import generate_complaints
from pipeline.embedder import Embedder
from pipeline.clusterer import Clusterer
from pipeline.sentiment import SentimentAnalyzer
from pipeline.labeler import ClusterLabeler
from pipeline.trend_detector import TrendDetector

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Setup OK — pipeline classes imported')


Setup OK — pipeline classes imported


## Cell 2 — Generate 500 synthetic complaints


In [2]:
df = generate_complaints(n=500, seed=SEED)
print(f'Generated {len(df)} complaints')
print()
print('Distribution by category:')
print(df['category'].value_counts().sort_index().to_string())
print()
print('Five sample complaints:')
for i, row in df.sample(5, random_state=SEED).iterrows():
    print(f'  [{row.category:22}] {row.complaint_text[:90]}')


Generated 500 complaints

Distribution by category:
category
billing_dispute         45
compressor_noise        70
cooling_inefficiency    90
electrical_tripping     50
gas_leak                25
installation_defect     50
noise_misc              50
service_delay           60
water_leakage           60

Five sample complaints:
  [electrical_tripping   ] There is some electrical issue, MCB trips the moment compressor starts.
  [electrical_tripping   ] The AC trips the MCB after running for around five minutes every time.
  [electrical_tripping   ] There is some electrical issue, MCB trips the moment compressor starts. AC repeatedly trip
  [installation_defect   ] The outdoor unit bracket is loose and shaking, looks like installation defect.
  [installation_defect   ] Indoor unit seedha nahi laga, isi liye paani tapak raha hai pura time.


## Cell 3 — Embed complaints


In [3]:
embedder = Embedder()
embeddings = embedder.encode_batch(df['complaint_text'].tolist())
print(f'Embedding shape: {embeddings.shape}')
assert embeddings.shape == (500, 384), f'unexpected shape {embeddings.shape}'

from sklearn.metrics.pairwise import cosine_similarity

cooling_pair = df[df.category == 'cooling_inefficiency'].head(2).index.tolist()
noise_idx = df[df.category == 'compressor_noise'].head(1).index.tolist()[0]

sim_same = float(cosine_similarity(
    embeddings[cooling_pair[0]:cooling_pair[0]+1],
    embeddings[cooling_pair[1]:cooling_pair[1]+1],
)[0, 0])
sim_cross = float(cosine_similarity(
    embeddings[cooling_pair[0]:cooling_pair[0]+1],
    embeddings[noise_idx:noise_idx+1],
)[0, 0])

print(f'cooling-cooling similarity:  {sim_same:.3f}')
print(f'cooling-noise   similarity:  {sim_cross:.3f}')
assert sim_same > sim_cross, 'same-topic must be more similar than cross-topic'


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-29 11:41:22 [debug    ] embedder_encode_batch          cache_hits=0 total=500 uncached=500


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding shape: (500, 384)
cooling-cooling similarity:  0.149
cooling-noise   similarity:  0.068


## Cell 4 — Cluster the embeddings


In [4]:
clusterer = Clusterer()
result = clusterer.fit(embeddings)

print(f'Found {result.n_clusters} clusters')
print(f'Noise: {result.noise_count} ({result.noise_pct:.1f}%)')
print(f'Silhouette: {result.silhouette_score:.3f}')
print()
print('Cluster sizes:')
for cid in sorted(result.cluster_sizes):
    print(f'  cluster {cid:>2}: {result.cluster_sizes[cid]:>3} complaints  fp={result.fingerprints[cid][:12]}')


/home/auxilus/hvac_venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


/home/auxilus/hvac_venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-04-29 11:41:33 [info     ] clusterer_fit_complete         n=500 n_clusters=11 noise_count=31 noise_pct=6.2 silhouette=0.837


Found 11 clusters
Noise: 31 (6.2%)
Silhouette: 0.837

Cluster sizes:
  cluster  0:  72 complaints  fp=cc4079db3e8d
  cluster  1:  38 complaints  fp=f3758af03d68
  cluster  2:  45 complaints  fp=b2717718160a
  cluster  3:  43 complaints  fp=348d25351e7e
  cluster  4:  35 complaints  fp=9766de332c9a
  cluster  5:  40 complaints  fp=fdbaebab6864
  cluster  6:  36 complaints  fp=5845fb7187d5
  cluster  7:  16 complaints  fp=76c31881b5fa
  cluster  8:  54 complaints  fp=516f64e36244
  cluster  9:  46 complaints  fp=bda8a8a381cd
  cluster 10:  44 complaints  fp=97b2838bd563


## Cell 5 — Plotly UMAP scatter


In [5]:
import plotly.express as px

viz_df = pd.DataFrame({
    'x': result.coords_2d[:, 0],
    'y': result.coords_2d[:, 1],
    'cluster': [str(c) if c != -1 else 'noise' for c in result.labels],
    'text': df['complaint_text'].str.slice(0, 80).tolist(),
})

fig = px.scatter(
    viz_df, x='x', y='y', color='cluster', hover_data=['text'],
    title='HVAC Complaint Clusters — UMAP 2D Projection',
)
fig.update_traces(marker=dict(size=7, opacity=0.8))
out_html = Path('cluster_scatter.html')
fig.write_html(str(out_html))
print(f'Saved scatter → {out_html.resolve()}')
fig.show()


Saved scatter → /home/auxilus/projects/HVAC/hvac-ml/notebooks/cluster_scatter.html


## Cell 6 — Sentiment scoring (VADER)


In [6]:
analyzer = SentimentAnalyzer()
sentiments = analyzer.score_batch(df['complaint_text'].tolist())
df['sentiment_score'] = [s.compound for s in sentiments]
df['sentiment_label'] = [s.label for s in sentiments]

print('Sentiment label distribution:')
print(df['sentiment_label'].value_counts().to_string())
print()
print('Top 5 most angry complaints (lowest compound):')
for _, row in df.nsmallest(5, 'sentiment_score').iterrows():
    print(f'  [{row.sentiment_score:+.3f}] {row.complaint_text[:90]}')


Sentiment label distribution:
sentiment_label
NORMAL      366
HIGH         91
CRITICAL     25
POSITIVE     18

Top 5 most angry complaints (lowest compound):
  [-0.954] PATHETIC SERVICE!!!! COOLING HAS BEEN VERY POOR SINCE LAST WEEK ALMOST NO COLD AIR COMES O
  [-0.881] WORST SERVICE!! INVOICE SHOWS CHARGES FOR PARTS THAT WERE NEVER ACTUALLY REPLACED OR INSTA
  [-0.880] PATHETIC SERVICE!!!! COOLING HAS BEEN VERY POOR SINCE LAST WEEK ALMOST NO COLD AIR COMES O
  [-0.871] WORST SERVICE!! A CONSTANT BUZZING SOUND FROM THE INDOOR UNIT IS VERY DISTURBING AT NIGHT!
  [-0.861] WORST SERVICE!! THE TEMPERATURE STAYS AT 30 DEGREES NO MATTER WHAT WE SET ON THE REMOTE!!


## Cell 7 — Auto-label clusters via Claude API

Requires `ANTHROPIC_API_KEY`. If unset, falls back to a topic-template label so
downstream cells still run.


In [7]:
df['cluster_id'] = result.labels

cluster_complaints: dict[int, list[str]] = {}
for cid in sorted(result.cluster_sizes):
    samples = df.loc[df.cluster_id == cid, 'complaint_text'].head(10).tolist()
    cluster_complaints[cid] = samples

labels: dict[int, str] = {}
if os.environ.get('ANTHROPIC_API_KEY'):
    labeler = ClusterLabeler()
    labels = labeler.label_all_clusters(cluster_complaints)
else:
    print('ANTHROPIC_API_KEY not set — using fallback category-vote labels')
    for cid, _ in cluster_complaints.items():
        cats = df.loc[df.cluster_id == cid, 'category'].mode()
        labels[cid] = cats.iloc[0].replace('_', ' ').title() if len(cats) else f'Cluster {cid}'

for cid, label in sorted(labels.items()):
    size = result.cluster_sizes[cid]
    avg_sent = float(df.loc[df.cluster_id == cid, 'sentiment_score'].mean())
    print(f'  Cluster {cid} (n={size:>3}): {label:<35}  avg sentiment {avg_sent:+.3f}')


ANTHROPIC_API_KEY not set — using fallback category-vote labels
  Cluster 0 (n= 72): Cooling Inefficiency                 avg sentiment -0.254
  Cluster 1 (n= 38): Electrical Tripping                  avg sentiment -0.181
  Cluster 2 (n= 45): Water Leakage                        avg sentiment -0.298
  Cluster 3 (n= 43): Installation Defect                  avg sentiment -0.170
  Cluster 4 (n= 35): Service Delay                        avg sentiment -0.318
  Cluster 5 (n= 40): Billing Dispute                      avg sentiment -0.242
  Cluster 6 (n= 36): Compressor Noise                     avg sentiment -0.134
  Cluster 7 (n= 16): Compressor Noise                     avg sentiment -0.276
  Cluster 8 (n= 54): Noise Misc                           avg sentiment -0.299
  Cluster 9 (n= 46): Cooling Inefficiency                 avg sentiment -0.062
  Cluster 10 (n= 44): Gas Leak                             avg sentiment -0.067


## Cell 8 — Week-over-week trend detection


In [8]:
detector = TrendDetector()
trends = detector.compute_trends(df[df.cluster_id != -1])

for t in sorted(trends, key=lambda x: x.growth_pct, reverse=True):
    flag = '🚨 EMERGING' if t.is_emerging else ''
    print(
        f'Cluster {t.cluster_id:>2}: {t.growth_pct:+7.1f}% WoW '
        f'(cur={t.current_week_count:>3}, prev={t.previous_week_count:>3}) | '
        f'Rs.{t.cost_exposure:>10,.0f} exposure {flag}'
    )


2026-04-29 11:41:34 [info     ] trend_detector_complete        clusters=11 emerging=6


Cluster  7:  +250.0% WoW (cur=  7, prev=  2) | Rs.    85,000 exposure 🚨 EMERGING
Cluster  8:  +250.0% WoW (cur=  7, prev=  2) | Rs.   127,500 exposure 🚨 EMERGING
Cluster 10:  +200.0% WoW (cur=  9, prev=  3) | Rs.   178,500 exposure 🚨 EMERGING
Cluster  9:  +150.0% WoW (cur=  5, prev=  2) | Rs.   127,500 exposure 🚨 EMERGING
Cluster  2:  +100.0% WoW (cur=  4, prev=  2) | Rs.   102,000 exposure 🚨 EMERGING
Cluster  3:   +33.3% WoW (cur=  4, prev=  3) | Rs.   153,000 exposure 🚨 EMERGING
Cluster  6:   +25.0% WoW (cur= 10, prev=  8) | Rs.   187,000 exposure 
Cluster  1:   -40.0% WoW (cur=  3, prev=  5) | Rs.   119,000 exposure 
Cluster  0:   -42.9% WoW (cur=  4, prev=  7) | Rs.   195,500 exposure 
Cluster  4:   -50.0% WoW (cur=  2, prev=  4) | Rs.    93,500 exposure 
Cluster  5:   -60.0% WoW (cur=  2, prev=  5) | Rs.   153,000 exposure 


## Cell 9 — Sprint 1 summary + exit-criteria assertions


In [9]:
rows = []
trend_by_cid = {t.cluster_id: t for t in trends}
for cid in sorted(result.cluster_sizes):
    t = trend_by_cid.get(cid)
    avg_sent = float(df.loc[df.cluster_id == cid, 'sentiment_score'].mean())
    rows.append({
        'Cluster': cid,
        'Label': labels.get(cid, f'cluster {cid}')[:30],
        'Count': result.cluster_sizes[cid],
        'Sent.': f'{avg_sent:+.2f}',
        'Growth': f'{t.growth_pct:+.0f}%' if t else '—',
        'Exposure': f'Rs.{t.cost_exposure:,.0f}' if t else '—',
        'Emerging': '🚨' if (t and t.is_emerging) else '',
    })
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print()

# ── Sprint 1 exit-criteria assertions ───────────────────────────────────
assert result.n_clusters >= 5, f'Expected ≥5 clusters, got {result.n_clusters}'
assert result.silhouette_score > 0.2, (
    f'Silhouette must be > 0.2, got {result.silhouette_score:.3f}'
)

critical_count = int((df.sentiment_label == 'CRITICAL').sum())
assert critical_count >= 2, f'Need ≥2 CRITICAL complaints, got {critical_count}'

# Delhi compressor noise spike: at least one cluster heavy in compressor_noise
# from Delhi must show > 100% WoW growth (the engineered spike).
spike_clusters = []
for cid in result.cluster_sizes:
    members = df[df.cluster_id == cid]
    if len(members) == 0:
        continue
    top_cat = members['category'].mode().iloc[0]
    delhi_share = (members['region'] == 'Delhi').mean()
    if top_cat == 'compressor_noise' and delhi_share >= 0.3:
        spike_clusters.append(cid)

assert spike_clusters, 'No compressor_noise/Delhi cluster found'
spike_growths = [trend_by_cid[c].growth_pct for c in spike_clusters if c in trend_by_cid]
assert any(g > 100 for g in spike_growths), (
    f'Delhi compressor cluster must show >100% WoW growth — got {spike_growths}'
)

print()
print('✅ Sprint 1 exit criteria PASSED')
print(f'   - {result.n_clusters} clusters found')
print(f'   - silhouette {result.silhouette_score:.3f}')
print(f'   - {critical_count} CRITICAL complaints')
print(f'   - Delhi compressor spike growths: {[round(g) for g in spike_growths]}')


 Cluster                Label  Count Sent. Growth   Exposure Emerging
       0 Cooling Inefficiency     72 -0.25   -43% Rs.195,500         
       1  Electrical Tripping     38 -0.18   -40% Rs.119,000         
       2        Water Leakage     45 -0.30  +100% Rs.102,000        🚨
       3  Installation Defect     43 -0.17   +33% Rs.153,000        🚨
       4        Service Delay     35 -0.32   -50%  Rs.93,500         
       5      Billing Dispute     40 -0.24   -60% Rs.153,000         
       6     Compressor Noise     36 -0.13   +25% Rs.187,000         
       7     Compressor Noise     16 -0.28  +250%  Rs.85,000        🚨
       8           Noise Misc     54 -0.30  +250% Rs.127,500        🚨
       9 Cooling Inefficiency     46 -0.06  +150% Rs.127,500        🚨
      10             Gas Leak     44 -0.07  +200% Rs.178,500        🚨


✅ Sprint 1 exit criteria PASSED
   - 11 clusters found
   - silhouette 0.837
   - 25 CRITICAL complaints
   - Delhi compressor spike growths: [25, 250]
